In [1]:
# A simple OOP gym nutrition system with unit-conversion, activity levels,
# multiple meal-plan options per goal, and time-to-goal estimation.

# Helper functions
def convert_lbs_to_kg(lbs):
    return lbs / 2.20462

def convert_ft_in_to_cm(feet, inches):
    return feet * 30.48 + inches * 2.54

def get_activity_multiplier(choice):
    mapping = {
        1: 1.2,
        2: 1.375,
        3: 1.55,
        4: 1.725,
        5: 1.9
    }
    return mapping.get(choice, 1.2)

def get_activity_label(choice):
    labels = {
        1: "Sedentary",
        2: "Lightly Active",
        3: "Moderately Active",
        4: "Very Active",
        5: "Extra Active"
    }
    return labels.get(choice, "Sedentary")

def safe_float(prompt, min_val=None):
    while True:
        try:
            val = float(input(prompt).strip())
            if min_val is not None and val <= min_val:
                print(f"Please enter a value greater than {min_val}.")
                continue
            return val
        except ValueError:
            print("Invalid number — try again.")

def safe_int(prompt, valid_set=None, min_val=None):
    while True:
        try:
            val = int(input(prompt).strip())
            if valid_set and val not in valid_set:
                print(f"Please enter one of: {valid_set}")
                continue
            if min_val is not None and val <= min_val:
                print(f"Please enter a value greater than {min_val}.")
                continue
            return val
        except ValueError:
            print("Invalid integer — try again.")

def convert_target_to_kg(amount, unit):
    unit = unit.lower()
    if unit in ("kg", "kilograms", "k"):
        return amount
    return convert_lbs_to_kg(amount)

# Parent Class
class Client:
    def __init__(self, name, age, weight_kg, height_cm, gender, activity_level):
        self.name = name
        self.age = age
        self.weight = weight_kg
        self.height = height_cm
        self.gender = gender.lower()
        self.activity_level = activity_level

    def calculate_bmr(self):
        """Mifflin-St Jeor equation with gender adjustment."""
        base = 10 * self.weight + 6.25 * self.height - 5 * self.age
        if self.gender in ("male", "m"):
            return base + 5
        return base - 161

    def daily_energy_expenditure(self):
        bmr = self.calculate_bmr()
        return bmr * get_activity_multiplier(self.activity_level)

# Child Classes
class WeightLossClient(Client):
    def calculate_calories(self):
        tdee = self.daily_energy_expenditure()
        return tdee - 500

    def available_plans(self):
        return {
            1: {
                "Breakfast": "Anda + Paratha (small) OR Daliya",
                "Lunch": "Grilled Chicken + Mixed Sabzi + 1 Roti",
                "Dinner": "Fish Curry + Salad",
                "Snacks": "1 Apple or Channay"
            },
            2: {
                "Breakfast": "Boiled Eggs + Brown Bread",
                "Lunch": "Daal + 1 Roti + Cucumber",
                "Dinner": "Chicken Handi (light oil) + Salad",
                "Snacks": "Nuts (small handful)"
            },
            3: {
                "Breakfast": "Oats in Milk + Banana (half)",
                "Lunch": "Chickpea Salad (Channa Chaat, low spice)",
                "Dinner": "Grilled Tikka (Chicken) + Veggies",
                "Snacks": "Yogurt"
            }
        }

class MuscleGainClient(Client):
    def calculate_calories(self):
        tdee = self.daily_energy_expenditure()
        return tdee + 300

    def available_plans(self):
        return {
            1: {
                "Breakfast": "3 Eggs + Paratha + Tea",
                "Lunch": "Chicken Biryani (moderate portion)",
                "Dinner": "Beef Qorma + 2 Rotis",
                "Snacks": "Peanut Butter Sandwich"
            },
            2: {
                "Breakfast": "Oats + Peanut Butter + Banana",
                "Lunch": "Chicken Karahi + Rice",
                "Dinner": "Daal + Chicken + Roti",
                "Snacks": "Lassi or Milkshake"
            },
            3: {
                "Breakfast": "Anda + Bread + Milk",
                "Lunch": "Grilled Chicken + Pasta",
                "Dinner": "Beef Kabab + Rice",
                "Snacks": "Dates + Nuts"
            }
        }

class MaintenanceClient(Client):
    def calculate_calories(self):
        return self.daily_energy_expenditure()

    def available_plans(self):
        return {
            1: {
                "Breakfast": "Omelette + Roti",
                "Lunch": "Chicken + Rice + Salad",
                "Dinner": "Fish + Sabzi",
                "Snacks": "Fruit + Nuts"
            },
            2: {
                "Breakfast": "Paratha (small) + Anda",
                "Lunch": "Paneer Wrap",
                "Dinner": "Chicken Curry + Roti",
                "Snacks": "Yogurt"
            },
            3: {
                "Breakfast": "Doodh Patti + Brown Bread",
                "Lunch": "Beef + Veggies",
                "Dinner": "Mixed Grill + Salad",
                "Snacks": "Fruit Bowl"
            }
        }
# Time-to-goal helpers

CALORIES_PER_KG = 7700

def estimate_time_months_kg_lose(kg, daily_deficit=500):
    if daily_deficit <= 0: return None
    return (CALORIES_PER_KG * kg) / daily_deficit / 30

def estimate_time_months_kg_gain(kg, daily_surplus=300):
    if daily_surplus <= 0: return None
    return (CALORIES_PER_KG * kg) / daily_surplus / 30

# Meal plan utilities

def print_meal_plan(plan):
    for meal, items in plan.items():
        print(f"  {meal}: {items}")

def select_plan_and_show(client_obj, calories):
    plans = client_obj.available_plans()

    print("\nAvailable meal plan options:\n")

    # Show all plans with full details
    for option_number, plan in plans.items():
        print(f"--- Option {option_number} ---")
        for meal, items in plan.items():
            print(f"  {meal}: {items}")
        print()  # empty line between options

    # Let user choose one
    choice = safe_int("Choose an option (1/2/3): ", valid_set={1,2,3})
    chosen = plans[choice]

    print("\nSelected meal plan:")
    print_meal_plan(chosen)
    return chosen


# main() keeps workflow organized

def main():
    print("=== Gym Nutrition System ===\n")
    name = input("Name: ").strip()
    age = safe_int("Age (years): ", min_val=10)

    print("\nWeight units: 1) kg  2) lbs")
    w_unit = safe_int("Choose weight unit (1 or 2): ", valid_set={1,2})
    if w_unit == 1:
        weight_kg = safe_float("Weight (kg): ", min_val=0)
    else:
        weight_lbs = safe_float("Weight (lbs): ", min_val=0)
        weight_kg = convert_lbs_to_kg(weight_lbs)

    print("\nHeight input: 1) centimeters  2) feet + inches")
    h_choice = safe_int("Choose height option (1 or 2): ", valid_set={1,2})
    if h_choice == 1:
        height_cm = safe_float("Height (cm): ", min_val=0)
    else:
        feet = safe_int("Feet: ", min_val=0)
        inches = safe_float("Inches: ", min_val=0)
        height_cm = convert_ft_in_to_cm(feet, inches)

    gender = input("\nGender (male/female): ").strip().lower()
    while gender not in ("male", "female", "m", "f"):
        print("Please enter 'male' or 'female'.")
        gender = input("Gender (male/female): ").strip().lower()

    print("\nActivity levels:\n 1) Sedentary\n 2) Lightly Active\n 3) Moderately Active\n 4) Very Active\n 5) Extra Active")
    activity = safe_int("Choose activity level (1-5): ", valid_set={1,2,3,4,5})

    print("\nGoals: loss | gain | maintain")
    goal = input("Enter goal: ").strip().lower()
    while goal not in ("loss", "gain", "maintain"):
        print("Choose 'loss', 'gain', or 'maintain'.")
        goal = input("Enter goal: ").strip().lower()

    if goal == "loss":
        client = WeightLossClient(name, age, weight_kg, height_cm, gender, activity)
    elif goal == "gain":
        client = MuscleGainClient(name, age, weight_kg, height_cm, gender, activity)
    else:
        client = MaintenanceClient(name, age, weight_kg, height_cm, gender, activity)

    calories = client.calculate_calories()
    calories = max(1200, calories)

    chosen_plan = select_plan_and_show(client, calories)

    months_needed = None
    if goal in ("loss", "gain"):
        print("\nDo you want to enter your target weight change in kg or lbs?")
        t_unit = input("Enter 'kg' or 'lbs': ").strip().lower()
        while t_unit not in ("kg", "lbs", "k", "l"):
            t_unit = input("Enter 'kg' or 'lbs': ").strip().lower()

        target_amount = safe_float(f"How many {t_unit} do you want to {'lose' if goal=='loss' else 'gain'}? ", min_val=0)
        target_kg = convert_target_to_kg(target_amount, t_unit)

        if goal == "loss":
            months_needed = estimate_time_months_kg_lose(target_kg)
        else:
            months_needed = estimate_time_months_kg_gain(target_kg)

    print("\n" + "-"*30)
    print("PERSONALIZED NUTRITION REPORT")
    print("-"*30)
    print(f"Name: {client.name}")
    print(f"Age: {client.age}")
    print(f"Gender: {client.gender}")
    print(f"Weight: {client.weight:.1f} kg")
    print(f"Height: {client.height:.1f} cm")
    print(f"Activity level: {get_activity_label(activity)}")
    print(f"Goal: {goal.capitalize()}")
    print(f"Daily Calorie Target: {int(calories)} kcal")
    print("\nSelected meal plan:")
    print_meal_plan(chosen_plan)

    if months_needed is not None:
        print(f"\nEstimated time to {goal} {target_amount} {t_unit}: ~{months_needed:.1f} months.")

    print("\nNote: These are estimates. Consult a nutrition professional for medical/clinical cases.")
    print("-"*30)

if __name__ == "__main__":
    main()



=== Gym Nutrition System ===


Weight units: 1) kg  2) lbs

Height input: 1) centimeters  2) feet + inches

Activity levels:
 1) Sedentary
 2) Lightly Active
 3) Moderately Active
 4) Very Active
 5) Extra Active

Goals: loss | gain | maintain

Available meal plan options:

--- Option 1 ---
  Breakfast: Anda + Paratha (small) OR Daliya
  Lunch: Grilled Chicken + Mixed Sabzi + 1 Roti
  Dinner: Fish Curry + Salad
  Snacks: 1 Apple or Channay

--- Option 2 ---
  Breakfast: Boiled Eggs + Brown Bread
  Lunch: Daal + 1 Roti + Cucumber
  Dinner: Chicken Handi (light oil) + Salad
  Snacks: Nuts (small handful)

--- Option 3 ---
  Breakfast: Oats in Milk + Banana (half)
  Lunch: Chickpea Salad (Channa Chaat, low spice)
  Dinner: Grilled Tikka (Chicken) + Veggies
  Snacks: Yogurt


Selected meal plan:
  Breakfast: Boiled Eggs + Brown Bread
  Lunch: Daal + 1 Roti + Cucumber
  Dinner: Chicken Handi (light oil) + Salad
  Snacks: Nuts (small handful)

Do you want to enter your target weight change i